# Capstone 07 — End-to-End Fine-Tuning Pipeline (Data to SFT to DPO to Serve): Interactive Visual Explorer

> An 8B model trained on your own data, DPO-aligned on your own preferences, quantized, speculative-decoded, and served at measurable $/1M tokens. The 2026 open stack is Axolotl v0.8, TRL 0.15, Unsloth for iteration, GPTQ/AWQ/GGUF for quantization, vLLM 0.7 with EAGLE-3 for serving. The capstone is to run the whole pipeline reproducibly — YAML in, served endpoint out — and publish a model card under the 2026 Model Openness Framework.

Welcome to the interactive companion notebook for **Capstone 07 — End-to-End Fine-Tuning Pipeline (Data to SFT to DPO to Serve)**.

In this notebook, you can interactively execute the lesson's raw implementation, plot state transformations, and run experiment variations.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# Configure plotting aesthetics
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 11


In [ ]:
"""End-to-end fine-tuning pipeline orchestrator scaffold.

The hard architectural primitive is a reproducible pipeline DAG: data hygiene
-> SFT -> preference tuning -> quantization -> serving -> eval -> model card,
where each stage is declaratively configured (YAML-ish dict here) and each
stage consumes the previous stage's artifact by content hash. This scaffold
models the DAG, the artifact manifest, and the contamination check.


In [ ]:
Run:  python main.py
"""

from __future__ import annotations

import hashlib
import json
import time
from dataclasses import dataclass, field
from typing import Callable

# ---------------------------------------------------------------------------
# artifact + manifest  --  content-hashed bookkeeping
# ---------------------------------------------------------------------------

@dataclass
class Artifact:
    name: str
    kind: str         # "dataset" | "checkpoint" | "quant" | "endpoint" | "report"
    payload: dict
    produced_by: str
    produced_at: float = field(default_factory=time.time)


In [ ]:
def content_hash(self) -> str:
        blob = json.dumps(self.payload, sort_keys=True, default=str).encode()
        return hashlib.sha256(blob).hexdigest()[:12]

@dataclass
class Manifest:
    artifacts: dict[str, Artifact] = field(default_factory=dict)

def add(self, a: Artifact) -> None:
        self.artifacts[a.name] = a

def get(self, name: str) -> Artifact:
        return self.artifacts[name]


In [ ]:
def summary(self) -> list[tuple[str, str, str, str]]:
        return [(a.name, a.kind, a.content_hash(), a.produced_by)
                for a in self.artifacts.values()]

# ---------------------------------------------------------------------------
# stages  --  each returns a new Artifact given prior manifest and config
# ---------------------------------------------------------------------------

Stage = Callable[[Manifest, dict], Artifact]


In [ ]:
def stage_data(m: Manifest, cfg: dict) -> Artifact:
    raw_n = cfg.get("raw_examples", 300_000)
    dedup_ratio = 0.94
    qual_ratio = 0.91
    pii_ratio = 0.995
    kept = int(raw_n * dedup_ratio * qual_ratio * pii_ratio)
    return Artifact("dataset", "dataset", {
        "raw_examples": raw_n,
        "after_dedup": int(raw_n * dedup_ratio),
        "after_quality": int(raw_n * dedup_ratio * qual_ratio),
        "after_pii_scrub": kept,
        "seed": cfg.get("seed", 7),
    }, produced_by="Datatrove+Nemotron-CC+Presidio")


In [ ]:
def stage_contamination(m: Manifest, cfg: dict) -> Artifact:
    ds = m.get("dataset")
    overlap = []
    for bench in ("MMLU-Pro", "MT-Bench-v2", "RewardBench-2"):
        # simulated MinHash check; real pipeline uses Datatrove MinHashLSH
        overlap.append({"bench": bench, "overlap_examples": 0})
    return Artifact("contamination_check", "report", {
        "dataset_hash": ds.content_hash(),
        "overlaps": overlap,
        "status": "clean" if all(o["overlap_examples"] == 0 for o in overlap) else "dirty",
    }, produced_by="minhash-lsh")


In [ ]:
def stage_sft(m: Manifest, cfg: dict) -> Artifact:
    ds = m.get("dataset")
    return Artifact("sft_checkpoint", "checkpoint", {
        "base": cfg["base_model"],
        "dataset_hash": ds.content_hash(),
        "epochs": 3,
        "val_loss": 1.03,
        "hours": 6.2,
        "gpus": 8,
    }, produced_by="axolotl v0.8 + ZeRO-3")

def stage_dpo(m: Manifest, cfg: dict) -> Artifact:
    sft = m.get("sft_checkpoint")
    return Artifact("dpo_checkpoint", "checkpoint", {
        "from": sft.content_hash(),
        "epochs": 1,
        "beta": 0.08,
        "hours": 1.7,
    }, produced_by="trl 0.15 DPO")


In [ ]:
def stage_quantize(m: Manifest, cfg: dict) -> Artifact:
    ckpt = m.get("dpo_checkpoint")
    return Artifact("quants", "quant", {
        "from": ckpt.content_hash(),
        "gptq_int4_gb": 4.6,
        "awq_int4_gb": 4.8,
        "gguf_q4_km_gb": 5.1,
    }, produced_by="gptq+awq+llama.cpp")

def stage_serve(m: Manifest, cfg: dict) -> Artifact:
    quants = m.get("quants")
    return Artifact("endpoint", "endpoint", {
        "backend": "vLLM 0.7 + EAGLE-3",
        "quant": "GPTQ-INT4-Marlin",
        "eagle_acceptance": 0.74,
        "p99_bs8_ms": 126,
        "tokens_per_sec_bs32": 6400,
        "dollars_per_mtokens": 0.28,
    }, produced_by="vllm+speculators")


In [ ]:
def stage_eval(m: Manifest, cfg: dict) -> Artifact:
    ckpt = m.get("dpo_checkpoint")
    return Artifact("eval_report", "report", {
        "from": ckpt.content_hash(),
        "mmlu_pro_delta": 3.2,
        "mt_bench_v2_delta": 0.41,
        "rewardbench2_delta": 0.08,
        "llama_guard_4_pass": 0.987,
    }, produced_by="lm-eval-harness")

def stage_model_card(m: Manifest, cfg: dict) -> Artifact:
    return Artifact("model_card", "report", {
        "standard": "MOF 2026",
        "data_license_declared": True,
        "training_config_hash": m.get("sft_checkpoint").content_hash(),
        "eval_attached": True,
        "safety_attached": True,
        "reproducibility_command": "./pipeline.sh config/llama3.3-8b-domainX.yaml",
    }, produced_by="mof-template")


In [ ]:
# ---------------------------------------------------------------------------
# DAG orchestrator  --  runs stages in order, snapshots manifest each step
# ---------------------------------------------------------------------------

PIPELINE: list[tuple[str, Stage]] = [
    ("data", stage_data),
    ("contamination", stage_contamination),
    ("sft", stage_sft),
    ("dpo", stage_dpo),
    ("quantize", stage_quantize),
    ("serve", stage_serve),
    ("eval", stage_eval),
    ("model_card", stage_model_card),
]


In [ ]:
def run_pipeline(cfg: dict) -> Manifest:
    m = Manifest()
    for name, stage_fn in PIPELINE:
        print(f"[{name:14s}] running...")
        art = stage_fn(m, cfg)
        m.add(art)
        print(f"[{name:14s}] -> artifact '{art.name}' hash={art.content_hash()}")
    return m

def main() -> None:
    cfg = {
        "base_model": "llama-3.3-8b",
        "raw_examples": 300_000,
        "seed": 7,
        "dpo_beta": 0.08,
    }
    print("=== fine-tuning pipeline run ===")
    m = run_pipeline(cfg)
    print()
    print("=== manifest ===")
    for name, kind, h, by in m.summary():
        print(f"  {name:18s} {kind:10s} {h} by {by}")
    print()
    print("=== eval report ===")
    print(json.dumps(m.get("eval_report").payload, indent=2))
    print()
    print("=== served endpoint ===")
    print(json.dumps(m.get("endpoint").payload, indent=2))


In [ ]:
if __name__ == "__main__":
    main()
